# 📄 PDF Chatbot using Retrieval-Augmented Generation (RAG)

A complete, self-contained Google Colab notebook that lets you upload a PDF and ask questions about it. Answers are grounded strictly in the document content, with page numbers cited. Run the cells top to bottom.

**Pipeline:** PDF → text extraction → word-based chunking (200 words, 40 overlap) → embeddings (`all-MiniLM-L6-v2`) → ChromaDB → similarity search (top 4) → grounded generation with `HuggingFaceH4/zephyr-7b-beta` (4-bit) → Gradio chat UI.

# Install Libraries

In [1]:
!pip install -q PyPDF2 sentence-transformers chromadb transformers accelerate bitsandbytes gradio torch


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 83.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7

# Imports

In [2]:
import os
import uuid
import warnings
from typing import List, Dict, Tuple, Optional

import torch
import gradio as gr
import chromadb
from chromadb.config import Settings
from PyPDF2 import PdfReader
from PyPDF2.errors import PdfReadError
from sentence_transformers import SentenceTransformer
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    pipeline,
)

warnings.filterwarnings("ignore")


# Load Embedding Model

In [3]:
# The embedding model is loaded exactly once and cached at module level so
# repeated calls (e.g. from within Gradio callbacks) do not reload it.
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"

_embedding_model: Optional[SentenceTransformer] = None


def get_embedding_model() -> SentenceTransformer:
    """Return a cached SentenceTransformer instance, loading it once."""
    global _embedding_model
    if _embedding_model is None:
        print(f"Loading embedding model: {EMBEDDING_MODEL_NAME} ...")
        _embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
        print("Embedding model loaded.")
    return _embedding_model


# Trigger the first (and only) load.
embedding_model = get_embedding_model()


Loading embedding model: all-MiniLM-L6-v2 ...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded.


# Load Zephyr Model

In [4]:
# The LLM is loaded exactly once and cached at module level. CUDA is detected
# automatically; if unavailable, a friendly warning is shown and the model is
# loaded on CPU in a slower, non-4-bit configuration (bitsandbytes 4-bit
# quantization requires a CUDA GPU).
LLM_MODEL_NAME = "HuggingFaceH4/zephyr-7b-beta"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if DEVICE != "cuda":
    print(
        "⚠️  WARNING: No CUDA GPU detected. Zephyr-7B is a large model and will "
        "be extremely slow (or may run out of memory) on CPU. In Google Colab, "
        "go to Runtime > Change runtime type > select a GPU (e.g. T4) for a "
        "usable experience."
    )
else:
    print(f"CUDA GPU detected: {torch.cuda.get_device_name(0)}")

_llm_model = None
_llm_tokenizer = None
_llm_pipeline = None


def get_llm_pipeline():
    """Return a cached text-generation pipeline for Zephyr, loading it once."""
    global _llm_model, _llm_tokenizer, _llm_pipeline

    if _llm_pipeline is not None:
        return _llm_pipeline

    print(f"Loading LLM: {LLM_MODEL_NAME} ...")

    _llm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_NAME)

    if DEVICE == "cuda":
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
        _llm_model = AutoModelForCausalLM.from_pretrained(
            LLM_MODEL_NAME,
            quantization_config=bnb_config,
            device_map="auto",
        )
    else:
        # CPU fallback: no 4-bit quantization available without a GPU.
        _llm_model = AutoModelForCausalLM.from_pretrained(
            LLM_MODEL_NAME,
            torch_dtype=torch.float32,
        )
        _llm_model.to("cpu")

    _llm_pipeline = pipeline(
        "text-generation",
        model=_llm_model,
        tokenizer=_llm_tokenizer,
    )

    print("LLM loaded.")
    return _llm_pipeline


# Trigger the first (and only) load.
llm_pipeline = get_llm_pipeline()


CUDA GPU detected: Tesla T4
Loading LLM: HuggingFaceH4/zephyr-7b-beta ...


config.json:   0%|          | 0.00/638 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.43k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  493kB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

LLM loaded.


# Helper Functions

General-purpose helpers used throughout the notebook (ChromaDB client setup, session management, etc.).

In [5]:
# A single, shared ChromaDB client for the whole notebook session.
chroma_client = chromadb.Client(Settings(anonymized_telemetry=False))

# Tracks the currently active session/collection so we don't recreate
# ChromaDB collections on every question — the same collection is reused
# for every question asked about a given uploaded PDF.
current_session: Dict[str, Optional[str]] = {
    "session_id": None,
    "collection_name": None,
}


def new_session_id() -> str:
    """Generate a fresh session id for a newly uploaded PDF."""
    return uuid.uuid4().hex


def get_or_create_collection(collection_name: str):
    """Get an existing Chroma collection or create it if it does not exist."""
    return chroma_client.get_or_create_collection(name=collection_name)


# PDF Extraction

In [6]:
def extract_pdf_pages(pdf_path: str) -> List[Tuple[int, str]]:
    """Extract text from every page of a PDF.

    Empty (whitespace-only) pages are skipped. Returns a list of
    (page_number, page_text) tuples, where page_number is 1-indexed.

    Raises:
        ValueError: if the PDF is corrupted, unreadable, or contains no
            extractable text at all.
    """
    if not pdf_path or not os.path.exists(pdf_path):
        raise ValueError("No PDF file was provided or the file could not be found.")

    try:
        reader = PdfReader(pdf_path)
    except PdfReadError as exc:
        raise ValueError(f"The PDF file appears to be corrupted: {exc}") from exc
    except Exception as exc:  # noqa: BLE001 - surface any parse failure clearly
        raise ValueError(f"Could not read the PDF file: {exc}") from exc

    if len(reader.pages) == 0:
        raise ValueError("The PDF file has no pages.")

    pages: List[Tuple[int, str]] = []
    for i, page in enumerate(reader.pages, start=1):
        try:
            text = page.extract_text() or ""
        except Exception:  # noqa: BLE001 - skip unreadable individual pages
            text = ""
        text = text.strip()
        if text:  # ignore empty pages
            pages.append((i, text))

    if not pages:
        raise ValueError(
            "No extractable text was found in this PDF. It may be a scanned "
            "image-only document without an OCR text layer."
        )

    return pages


# Chunking

In [7]:
def chunk_text(
    text: str,
    page_number: int,
    chunk_size: int = 200,
    overlap: int = 40,
) -> List[Dict]:
    """Split text into overlapping word-based chunks.

    Splitting is done strictly by words (never by raw characters), using a
    sliding window of `chunk_size` words that advances by
    `chunk_size - overlap` words each step.

    Args:
        text: The raw text of a single page.
        page_number: The 1-indexed page number this text came from.
        chunk_size: Number of words per chunk.
        overlap: Number of words shared between consecutive chunks.

    Returns:
        A list of dicts with keys: "text" and "page".
    """
    words = text.split()
    if not words:
        return []

    step = max(chunk_size - overlap, 1)
    chunks: List[Dict] = []

    for start in range(0, len(words), step):
        window = words[start:start + chunk_size]
        if not window:
            break
        chunks.append({
            "text": " ".join(window),
            "page": page_number,
        })
        if start + chunk_size >= len(words):
            break

    return chunks


def chunk_pdf_pages(pages: List[Tuple[int, str]]) -> List[Dict]:
    """Chunk every page of an extracted PDF, keeping page numbers attached."""
    all_chunks: List[Dict] = []
    for page_number, page_text in pages:
        all_chunks.extend(chunk_text(page_text, page_number))
    return all_chunks


# Embeddings

In [8]:
def embed_texts(texts: List[str]) -> List[List[float]]:
    """Embed a list of texts using the cached sentence-transformers model."""
    model = get_embedding_model()
    embeddings = model.encode(
        texts,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=True,  # cosine similarity via normalized dot product
    )
    return embeddings.tolist()


# ChromaDB

In [9]:
def index_pdf(pdf_path: str) -> Tuple[str, int]:
    """Full indexing pipeline for one uploaded PDF.

    Extracts text, chunks it, embeds every chunk, and stores everything in a
    brand-new ChromaDB collection named `col_<session_id>`. Each uploaded PDF
    gets its own collection.

    Returns:
        (collection_name, number_of_chunks_indexed)
    """
    pages = extract_pdf_pages(pdf_path)
    chunks = chunk_pdf_pages(pages)

    if not chunks:
        raise ValueError("No content could be chunked from this PDF.")

    session_id = new_session_id()
    collection_name = f"col_{session_id}"
    collection = get_or_create_collection(collection_name)

    texts = [c["text"] for c in chunks]
    embeddings = embed_texts(texts)

    ids = [f"{session_id}_chunk_{i}" for i in range(len(chunks))]
    metadatas = [{"page": c["page"], "chunk_id": ids[i]} for i, c in enumerate(chunks)]

    collection.add(
        ids=ids,
        embeddings=embeddings,
        documents=texts,
        metadatas=metadatas,
    )

    current_session["session_id"] = session_id
    current_session["collection_name"] = collection_name

    return collection_name, len(chunks)


# Retrieval

In [10]:
def retrieve_top_chunks(question: str, top_k: int = 4) -> List[Dict]:
    """Embed the question and retrieve the top-k most similar chunks.

    Returns:
        A list of dicts with keys: "text" and "page", ordered by
        decreasing similarity. Empty list if no collection is active or
        nothing is found.
    """
    collection_name = current_session.get("collection_name")
    if not collection_name:
        return []

    collection = get_or_create_collection(collection_name)

    question_embedding = embed_texts([question])[0]

    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=top_k,
    )

    documents = results.get("documents", [[]])[0]
    metadatas = results.get("metadatas", [[]])[0]

    retrieved = [
        {"text": doc, "page": meta.get("page")}
        for doc, meta in zip(documents, metadatas)
    ]
    return retrieved


# Prompt Builder

In [11]:
PROMPT_TEMPLATE = """You are a helpful assistant.

Answer ONLY using the context below.

If the answer is not contained in the context, say you couldn't find it.

Never make up information.

Always mention which page(s) were used.

Context:
{retrieved_chunks}

Question:
{user_question}

Answer:"""


def build_prompt(question: str, retrieved_chunks: List[Dict]) -> str:
    """Build the grounded RAG prompt from retrieved chunks and the question."""
    context_blocks = [
        f"[Page {chunk['page']}] {chunk['text']}" for chunk in retrieved_chunks
    ]
    context = "\n\n".join(context_blocks)
    return PROMPT_TEMPLATE.format(retrieved_chunks=context, user_question=question)


# LLM Response

In [12]:
NOT_FOUND_MESSAGE = "I couldn't find that information in the uploaded document."


def generate_answer(question: str) -> str:
    """Run the full retrieve -> prompt -> generate pipeline for one question.

    Handles all edge cases: no PDF indexed yet, no chunks retrieved, and
    overly long questions.
    """
    if not question or not question.strip():
        return "Please enter a question."

    if len(question.split()) > 200:
        return (
            "Your question is quite long. Please shorten it to under 200 "
            "words so it can be processed effectively."
        )

    if not current_session.get("collection_name"):
        return "Please upload and index a PDF before asking a question."

    retrieved_chunks = retrieve_top_chunks(question, top_k=4)

    if not retrieved_chunks:
        return NOT_FOUND_MESSAGE

    prompt = build_prompt(question, retrieved_chunks)

    llm = get_llm_pipeline()
    outputs = llm(
        prompt,
        max_new_tokens=256,
        do_sample=False,
        temperature=0.1,
        return_full_text=False,
        pad_token_id=llm.tokenizer.eos_token_id,
    )

    answer = outputs[0]["generated_text"].strip()

    if not answer:
        return NOT_FOUND_MESSAGE

    return answer


# Gradio App

In [24]:
def handle_pdf_upload(pdf_file) -> str:
    """Gradio callback: index an uploaded PDF and report status."""
    if pdf_file is None:
        return "⚠️ No PDF uploaded. Please choose a file first."

    pdf_path = pdf_file if isinstance(pdf_file, str) else pdf_file.name

    try:
        collection_name, num_chunks = index_pdf(pdf_path)
    except ValueError as exc:
        return f"❌ {exc}"
    except Exception as exc:  # noqa: BLE001 - surface unexpected errors clearly
        return f"❌ Unexpected error while indexing PDF: {exc}"

    return (
        f"✅ PDF indexed successfully. ({num_chunks} chunks stored in "
        f"collection `{collection_name}`)"
    )


def handle_question(question: str, history):
    """Gradio callback: answer a question and append it to the chat history."""

    history = history or []

    if not current_session.get("collection_name"):
        history.append({
            "role": "user",
            "content": question
        })
        history.append({
            "role": "assistant",
            "content": "⚠️ Please upload and index a PDF first."
        })
        return history, ""

    try:
        answer = generate_answer(question)
    except Exception as exc:
        answer = f"❌ An error occurred while generating the answer:\n{exc}"

    history.append({
        "role": "user",
        "content": question
    })

    history.append({
        "role": "assistant",
        "content": answer
    })

    return history, ""


with gr.Blocks(title="📄 PDF Chatbot using RAG") as demo:
    gr.Markdown("# 📄 PDF Chatbot using RAG")
    gr.Markdown(
        "Upload a PDF, wait for indexing to finish, then ask questions. "
        "Answers are grounded strictly in the document and cite page numbers."
    )

    with gr.Row():
        with gr.Column(scale=1):
            pdf_upload = gr.File(label="Upload PDF", file_types=[".pdf"])
            status_box = gr.Textbox(label="Status", interactive=False)
        with gr.Column(scale=2):
            chatbot = gr.Chatbot(
                label="Chat History",
                height=450,
            )
            question_box = gr.Textbox(
                label="Ask a question about the PDF",
                placeholder="e.g. What is the main conclusion of this document?",
            )
            ask_button = gr.Button("Ask", variant="primary")

    pdf_upload.change(fn=handle_pdf_upload, inputs=pdf_upload, outputs=status_box)
    ask_button.click(
        fn=handle_question,
        inputs=[question_box, chatbot],
        outputs=[chatbot, question_box],
    )
    question_box.submit(
        fn=handle_question,
        inputs=[question_box, chatbot],
        outputs=[chatbot, question_box],
    )

demo.launch(
    debug=True,
    share=True
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://4a8e9ac562a51f7d22.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://4a8e9ac562a51f7d22.gradio.live
